# 03 results

Steps 8, 10-11 - held-out GPT-2 test, the PDS diagnostic, Gemma validation.

Run only after the config hash in DECISIONS.md is frozen.

## Настройка

In [15]:
%load_ext autoreload
%autoreload 2

import json
import sys

import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

from steering import (corruptions, denoiser, generate, hooks, interventions, io, judge,
                      metrics, spaces, stats, vectors)

MODEL, LAYER = "gpt2", 6
DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(MODEL)
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL).eval().to(DEVICE)

sae = vectors.load_sae("gpt2-small-resid-post-v5-128k", "blocks.6.hook_resid_post",
                        layer=LAYER, d_model=768, device=DEVICE)
decoder = sae.W_dec.detach()

split = vectors.load_split(io.RESULTS / "feature_splits_gpt2.json")
center = spaces.should_center(MODEL)

prompts = json.loads((io.REPO_ROOT / "configs" / "prompts_neutral_32.json").read_text())["prompts"]
enc = tokenizer(prompts, return_tensors="pt", padding=True).to(DEVICE)

with hooks.ResidualHook(model, layer=LAYER, capture=True) as _h, torch.no_grad():
    model(**enc)
scale = spaces.activation_scale(_h.captured[0], attention_mask=enc["attention_mask"].cpu(), exclude_sink=True)


# The frozen method (DECISIONS D6) and the pre-declared judge subset -- both read back from
# disk, not re-derived, so this notebook uses exactly what was frozen, not a fresh guess at it.
frozen_config = json.loads((io.RESULTS / "frozen_method_config_gpt2.json").read_text())
judge_subset = json.loads((io.RESULTS / "judge_subset_gpt2.json").read_text())
R_GRID = frozen_config["r_grid"]
JUDGE_CONCEPTS = judge_subset["concepts"]
JUDGE_R = judge_subset["r_values"]

# Already-trained D4 checkpoints from Step 5's confirmation (seeds 0 and 1) -- Step 8 needs a
# third seed on top of these, not three from scratch.
d4_seed0 = denoiser.load_denoiser(io.ARTIFACTS / "denoiser_d4_seed0.pt", device=DEVICE)
d4_seed1 = denoiser.load_denoiser(io.ARTIFACTS / "denoiser_d4_seed1.pt", device=DEVICE)
d1_seed0 = denoiser.load_denoiser(io.ARTIFACTS / "denoiser_d1_seed0.pt", device=DEVICE)

j = judge.OllamaJudge(cache_dir=io.ARTIFACTS / "judge_cache")

print(f"python  {sys.version.split()[0]}")
print(f"device  {DEVICE}")
print(f"split   DEV {len(split.dev)}, TEST {len(split.test)}, fingerprint {split.fingerprint()}")
print(f"frozen  config hash {io.config_hash({**frozen_config, 'version': 1})}")
print(f"judge   {len(JUDGE_CONCEPTS)} concepts, r={JUDGE_R}")
print(f"r_grid  {R_GRID}")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

python  3.12.0
device  mps
split   DEV 35, TEST 70, fingerprint ce65a9496b253ceb
frozen  config hash 49db404c4e308bc0
judge   40 concepts, r=[0.2, 0.4, 0.6, 0.8]
r_grid  [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]


In [16]:
activations = io.run_or_load(
    "clean_activations_gpt2", {"model": "gpt2", "layer": LAYER, "n_docs": 3000,
                               "max_length": 64, "max_tokens": 200_000, "version": 1},
    lambda: (_ for _ in ()).throw(RuntimeError("cache missing -- rerun Step 4")),
    where="artifacts",
)
d4_corruption = corruptions.FullPoolRank1(decoder, split)
d1_corruption = corruptions.Gaussian(sigma_min=0.05, sigma_max=3.0)
print(f"activations {activations.shape}")

cached  clean_activations_gpt2.pt  (6be99263c06aee12)
activations torch.Size([186699, 768])


## Шаг 8. Сравниваем подходы

In [6]:
STEPS = frozen_config["training"]["steps"]
BATCH_SIZE = frozen_config["training"]["batch_size"]
LR = frozen_config["training"]["lr"]

d4_seed2, history2 = denoiser.train_denoiser(
    activations, d4_corruption, d_model=768, activation_scale=scale, center=center,
    steps=STEPS, batch_size=BATCH_SIZE, lr=LR, seed=2, device=DEVICE, log_every=STEPS // 10,
)
denoiser.save_denoiser(d4_seed2, io.ARTIFACTS / "denoiser_d4_seed2.pt",
                       extra={"corruption": d4_corruption.describe(), "seed": 2, "steps": STEPS})
print(f"D4 seed=2: final train loss {history2[-1]['loss']:.4f}")

d4_seeds = {0: d4_seed0, 1: d4_seed1, 2: d4_seed2}

D4 seed=2: final train loss 0.1907


In [7]:
all_concepts = sorted(set(split.dev) | set(split.test))
descriptions = vectors.describe_features(sae.cfg.metadata.neuronpedia_id, all_concepts,
                                         cache_dir=io.ARTIFACTS / "neuronpedia_cache")
descriptions_df = pd.DataFrame([{"feature_id": fid, **info} for fid, info in descriptions.items()])

neuronpedia:   0%|          | 0/105 [00:00<?, ?it/s]

In [8]:
io.run_or_load("concept_descriptions_gpt2",
               {"n_concepts": len(all_concepts), "split_fingerprint": split.fingerprint(),
                "version": 1}, lambda: descriptions_df)
print(f"{len(descriptions)} descriptions ({sum(1 for i in descriptions.values() if i['token_level'])} token-level)")

cached  concept_descriptions_gpt2.csv  (9b617ede50b2803e)
105 descriptions (0 token-level)


In [12]:
test_config = {"model": "gpt2", "layer": LAYER, "split_fingerprint": split.fingerprint(),
              "r_grid": R_GRID, "methods": [*ARMS, "B0"], "n_prompts": len(prompts),
              "max_new_tokens": 32, "seed": 0, "version": 1}
test_config

{'model': 'gpt2',
 'layer': 6,
 'split_fingerprint': 'ce65a9496b253ceb',
 'r_grid': [0.0, 0.2, 0.4, 0.6, 0.8, 1.0],
 'methods': ['B1', 'B2', 'D1', 'D4_seed0', 'D4_seed1', 'D4_seed2', 'B0'],
 'n_prompts': 32,
 'max_new_tokens': 32,
 'seed': 0,
 'version': 1}

In [ ]:
from tqdm import tqdm
ARMS = {
    "B1": lambda v, r: interventions.AdditiveSteering(v, r=r, scale=scale),
    "B2": lambda v, r: interventions.NormMatchedSteering(v, r=r, scale=scale),
    "D1": lambda v, r: interventions.DenoisedSteering(v, r=r, scale=scale, model=d1_seed0,
                                                       corruption_description=d1_corruption.describe()),
}
for seed, m in d4_seeds.items():
    ARMS[f"D4_seed{seed}"] = lambda v, r, m=m: interventions.DenoisedSteering(
        v, r=r, scale=scale, model=m, corruption_description=d4_corruption.describe())

test_directions = vectors.steering_directions(sae, split.test)
test_summary_rows, test_detail_rows = [], []

def record_test(method, r, fid, texts, nll):
    valid = ~nll.isnan()
    ppl = float(nll[valid].mean().exp()) if valid.any() else float("nan")
    score = metrics.sae_concept_score(model, tokenizer, sae, fid, LAYER, prompts, texts,
                                      device=DEVICE, center=center)
    test_summary_rows.append({"method": method, "r": r, "feature_id": fid, "ppl": ppl,
                              "dist_2": metrics.distinct_n(texts, 2),
                              "repetition_4": metrics.repetition_rate(texts, 4),
                              "concept_mean_act": score["mean_act"],
                              "concept_fire_rate": score["fire_rate"]})
    for i, (p, c, n) in enumerate(zip(prompts, texts, nll.tolist())):
        test_detail_rows.append({"method": method, "r": r, "feature_id": fid, "prompt_idx": i,
                                 "prompt": p, "continuation": c, "nll": n})

print("generating B0 baseline...")
b0 = generate.generate(model, tokenizer, prompts, interventions.NoSteering(), layer=LAYER,
                       device=DEVICE, max_new_tokens=32, batch_size=16, seed=0)
print("scoring B0 reference NLL...")
b0_nll = metrics.reference_nll(model, tokenizer, prompts, b0.texts, device=DEVICE)
print("computing B0 concept scores per TEST concept...")
for fid in tqdm(split.test, desc="B0 concept scores"):
    record_test("B0", 0.0, fid, b0.texts, b0_nll)

for arm_name, make_iv in ARMS.items():
    pairs = list(zip(split.test, test_directions))
    for fid, v in tqdm(pairs, desc=arm_name):
        for r in R_GRID[1:]:
            iv = make_iv(v, r)
            out = generate.generate(model, tokenizer, prompts, iv, layer=LAYER, device=DEVICE,
                                    max_new_tokens=32, batch_size=16, seed=0)
            nll = metrics.reference_nll(model, tokenizer, prompts, out.texts, device=DEVICE)
            record_test(arm_name, r, fid, out.texts, nll)

In [13]:
test_config = {"model": "gpt2", "layer": LAYER, "split_fingerprint": split.fingerprint(),
              "r_grid": R_GRID, "methods": [*ARMS, "B0"], "n_prompts": len(prompts),
              "max_new_tokens": 32, "seed": 0, "version": 1}
test_summary_df = io.run_or_load("test_sweep_summary_gpt2", test_config,
                                 lambda: pd.DataFrame(test_summary_rows))
test_detail_df = io.run_or_load("test_sweep_detail_gpt2", test_config,
                                lambda: pd.DataFrame(test_detail_rows), where="artifacts")
print(test_summary_df.groupby(["method", "r"])[["ppl", "dist_2", "concept_mean_act"]].mean().round(4))

cached  test_sweep_summary_gpt2.csv  (a751ab56afa4d0e0)
cached  test_sweep_detail_gpt2.csv  (a751ab56afa4d0e0)
                    ppl  dist_2  concept_mean_act
method   r                                       
B0       0.0    48.8763  0.9726            0.0008
B1       0.2    57.0440  0.9663            0.0120
         0.4   109.8359  0.9540            0.0557
         0.6   253.3071  0.9366            0.0835
         0.8   575.3808  0.9190            0.0977
         1.0   892.7145  0.9018            0.1118
B2       0.2    56.9811  0.9669            0.0111
         0.4   105.9956  0.9535            0.0509
         0.6   269.5028  0.9333            0.0748
         0.8   627.6476  0.9062            0.0779
         1.0   994.9547  0.8795            0.0675
D1       0.2    66.4922  0.9669            0.0115
         0.4   179.2188  0.9638            0.0688
         0.6   526.1443  0.9591            0.1507
         0.8  1181.2279  0.9556            0.1959
         1.0  1957.9038  0.9574        

In [18]:
j = judge.OllamaJudge(model="gemma4:26b", cache_dir=io.ARTIFACTS / "judge_cache", max_workers=8)

JUDGE_METHODS = ["B0", "B1", "B2", "D1", "D4_seed0"]

def draw_judge_prompt_subset():
    rng = torch.Generator().manual_seed(0)
    order = torch.randperm(len(prompts), generator=rng).tolist()
    return {"seed": 0, "n_prompts": 8, "prompt_idx": sorted(order[:8])}

judge_prompt_subset = io.run_or_load(
    "judge_prompt_subset_gpt2",
    {"n_total_prompts": len(prompts), "n_selected": 8, "seed": 0, "version": 1},
    draw_judge_prompt_subset,
)
JUDGE_PROMPT_IDX_STAGE1 = judge_prompt_subset["prompt_idx"]
JUDGE_PROMPT_IDX_STAGE2 = [i for i in range(len(prompts)) if i not in JUDGE_PROMPT_IDX_STAGE1]
print(f"stage 1: {len(JUDGE_PROMPT_IDX_STAGE1)} prompts {JUDGE_PROMPT_IDX_STAGE1}")
print(f"stage 2: {len(JUDGE_PROMPT_IDX_STAGE2)} remaining prompts")

computed judge_prompt_subset_gpt2.json  (4f582c0112923b37)
stage 1: 8 prompts [9, 10, 12, 19, 25, 28, 29, 31]
stage 2: 24 remaining prompts


In [19]:
def score_judge_subset(prompt_idx_list, label):
    is_b0 = test_detail_df["method"] == "B0"
    subset = test_detail_df[
        test_detail_df["feature_id"].isin(JUDGE_CONCEPTS)
        & test_detail_df["prompt_idx"].isin(prompt_idx_list)
        & test_detail_df["method"].isin(JUDGE_METHODS)
        & (is_b0 & (test_detail_df["r"] == 0.0) | ~is_b0 & test_detail_df["r"].isin(JUDGE_R))
    ].copy()

    unique_pairs = subset[["prompt", "continuation"]].drop_duplicates().reset_index(drop=True)
    print(f"{label}: {len(subset)} rows, {len(unique_pairs)} unique (prompt, continuation) pairs")
    coherence_scores = j.score_many(unique_pairs["prompt"].tolist(),
                                    unique_pairs["continuation"].tolist(),
                                    rubric="coherence", progress=True)
    coherence_map = dict(zip(zip(unique_pairs["prompt"], unique_pairs["continuation"]),
                             coherence_scores))
    subset["judge_coherence"] = [coherence_map[(p, c)]
                                 for p, c in zip(subset["prompt"], subset["continuation"])]

    subset["judge_concept"] = float("nan")
    for fid in tqdm(JUDGE_CONCEPTS, desc=f"{label}: concept"):
        mask = subset["feature_id"] == fid
        description = descriptions.get(fid, {}).get("description", "")
        if not description:
            raise ValueError(f"missing description for feature {fid}")
        subset.loc[mask, "judge_concept"] = j.score_many(
            subset.loc[mask, "prompt"].tolist(), subset.loc[mask, "continuation"].tolist(),
            rubric="concept", concept=description, progress=False)

    return subset[["method", "r", "feature_id", "prompt_idx",
                   "judge_coherence", "judge_concept"]].reset_index(drop=True)

In [20]:
judge_stage1_df = io.run_or_load(
    "test_judge_stage1_gpt2",
    {"methods": JUDGE_METHODS, "concepts": JUDGE_CONCEPTS, "r_values": JUDGE_R,
     "prompt_idx": JUDGE_PROMPT_IDX_STAGE1, "judge_model": j.model, "version": 1},
    lambda: score_judge_subset(JUDGE_PROMPT_IDX_STAGE1, "stage1"),
)
print(j.stats())
print(judge_stage1_df.groupby(["method", "r"])[["judge_coherence", "judge_concept"]].mean().round(1))

stage1: 5440 rows, 4616 unique (prompt, continuation) pairs


judge:coherence:   0%|          | 0/4616 [00:00<?, ?it/s]

stage1: concept: 100%|█████████████████████████| 40/40 [22:43<00:00, 34.08s/it]

computed test_judge_stage1_gpt2.csv  (45ed9c05758832f4)
{'calls': 9553, 'cache_hits': 503}
              judge_coherence  judge_concept
method   r                                  
B0       0.0             45.6            9.4
B1       0.2             42.3           17.7
         0.4             32.9           36.7
         0.6             20.5           37.3
         0.8             12.9           33.8
B2       0.2             41.7           17.1
         0.4             32.1           35.7
         0.6             18.2           33.7
         0.8             10.4           27.4
D1       0.2             41.7           19.1
         0.4             28.9           37.3
         0.6             16.9           43.4
         0.8             10.9           43.3
D4_seed0 0.2             38.8           19.4
         0.4             29.4           34.6
         0.6             18.4           45.3
         0.8             11.1           46.6


In [ ]:
import matplotlib.pyplot as plt

# Aggregate judge results by method / steering strength
judge_agg = (
    judge_stage1_df
    .groupby(["method", "r"], as_index=False)
    .agg(
        coherence=("judge_coherence", "median"),
        concept=("judge_concept", "median"),
        n=("feature_id", "nunique"),
    )
)


display(
    judge_agg
    .sort_values(["method", "r"])
    .round(2)
)


In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

# Each method trajectory
for method, sub in judge_agg.groupby("method"):
    sub = sub.sort_values("r")

    ax.plot(
        sub["coherence"],
        sub["concept"],
        marker="o",
        label=method,
        alpha=0.8,
    )

    for _, row in sub.iterrows():
        ax.annotate(
            f"r={row['r']:g}",
            (row["coherence"], row["concept"]),
            xytext=(4, 4),
            textcoords="offset points",
            fontsize=8,
        )


ax.set_xlabel("Judge coherence ↑")
ax.set_ylabel("Judge concept score ↑")
ax.set_title("GPT-2 TEST: judge Pareto frontier")
ax.grid(alpha=0.25)
ax.legend()

plt.show()

In [ ]:
judge_stage2_df = io.run_or_load(
    "test_judge_stage2_gpt2",
    {"methods": JUDGE_METHODS, "concepts": JUDGE_CONCEPTS, "r_values": JUDGE_R,
     "prompt_idx": JUDGE_PROMPT_IDX_STAGE2, "judge_model": j.model, "version": 1},
    lambda: score_judge_subset(JUDGE_PROMPT_IDX_STAGE2, "stage2"),
)

judge_df = pd.concat([judge_stage1_df, judge_stage2_df], ignore_index=True)
io.run_or_load("test_judge_gpt2",
               {"methods": JUDGE_METHODS, "concepts": JUDGE_CONCEPTS, "r_values": JUDGE_R,
                "n_prompts": len(prompts), "judge_model": j.model, "version": 1},
               lambda: judge_df)
print(j.stats())
print(judge_df.groupby(["method", "r"])[["judge_coherence", "judge_concept"]].mean().round(1))

stage2: 16320 rows, 13935 unique (prompt, continuation) pairs


judge:coherence:   0%|          | 0/13935 [00:00<?, ?it/s]

stage2: concept:  15%|███▍                   | 6/40 [10:35<1:00:34, 106.90s/it]

In [33]:
comparisons = {}
for baseline in ("B1", "B2", "D1"):
    arm_df = judge_df[judge_df.method == "D4_seed0"].rename(columns={"feature_id": "unit"})
    base_df = judge_df[judge_df.method == baseline].rename(columns={"feature_id": "unit"})

    quality_at_concept = stats.matched_comparison(
        pd.concat([arm_df.assign(method="D4"), base_df.assign(method=baseline)]),
        "D4", baseline, unit_col="unit", x="judge_concept", y="judge_coherence",
        targets=(25, 50, 75), seed=0)
    concept_at_quality = stats.matched_comparison(
        pd.concat([arm_df.assign(method="D4"), base_df.assign(method=baseline)]),
        "D4", baseline, unit_col="unit", x="judge_coherence", y="judge_concept",
        targets=(50, 70, 90), seed=0)
    band = stats.matched_band(arm_df.assign(method="D4"), base_df.assign(method=baseline),
                              unit_col="unit", x="judge_coherence", y="judge_concept",
                              n_boot=4000, n_grid=24, seed=0)
    comparisons[baseline] = {"quality_at_concept": quality_at_concept,
                             "concept_at_quality": concept_at_quality, "band": band}
    print(f"\n=== D4 vs {baseline} ===")
    print("concept at matched quality:\n", concept_at_quality.round(3))

pvalues = {f"{b}_r{row.target}": row.p for b, c in comparisons.items()
          for row in c["concept_at_quality"].itertuples()}
print("\nHolm-corrected p-values:", stats.holm(pvalues))

io.run_or_load("headline_comparison_gpt2",
               {"baselines": ["B1", "B2", "D1"], "targets_concept": [25, 50, 75],
                "targets_quality": [50, 70, 90], "version": 1},
               lambda: pd.concat([c["concept_at_quality"].assign(baseline=b)
                                  for b, c in comparisons.items()]))

NameError: name 'judge_df' is not defined